# Meta-control model family comparison (MATLAB → structured notes)

This notebook compiles and compares **7** closely related MATLAB models you provided:

- NoInhibition (Go-only)
- NoInhibitionNoGo (Go+NoGo)
- InhibitionRecall (Go-only)
- InhibitionRecallNoGo (Go+NoGo)
- InhibitionHabitualNoGo (Go+NoGo)
- RecallOverride (Go-only)
- RecallOverrideNoGo (Go+NoGo)

It focuses on: (i) shared structure, (ii) what *exactly* differs across variants, and (iii) model-specific pseudocode that mirrors the original implementation (variable names preserved where possible).


---
## Task

All models implement a cost–benefit meta-control policy over **AX-CPT-like** trial types.

### Trial-types
- **Go-only** variants define 4 trial types: `AX, AY, BX, BY` via:
  - `cues={'A','A','B','B'}; probes={'X','Y','X','Y'};`
- **NoGo** variants add 2 trial types with `probe='N'` (NoGo): `AN, BN`, giving 6 types total.

### Probabilities & conditioning
All variants start from:
- `p_cue_probe = [pAX, pAY, pBX, pBY]` (and optionally `pAN, pBN`)
- Construct conditional vectors like `p_given_A`, `p_given_B`, `p_given_X`, `p_given_Y` (and `p_given_N` in NoGo).

### Accuracy
Three accuracy quantities computed for each trial type `s` across all models:
1. `p_correct_habit(s)`: accuracy under a habitual policy (with slips `p_slip`)
2. `p_correct_proactive(s)`: accuracy when proactively setting an intention (degraded by load via `lambda*(load-1)`)
3. `p_correct_recall(s)`: accuracy if recall is executed (depends on `m` and load term)

Then define the **benefits** over habit:
- `delta_acc = p_correct_proactive - p_correct_habit`
- `delta_acc_recall = p_correct_recall - p_correct_habit`

And compress those into **expected benefits *conditional* on cue/probe**:
- `E_delta_acc_A, E_delta_acc_B, E_delta_acc_X, E_delta_acc_Y`
- `E_delta_acc_recall_A, ... , E_delta_acc_recall_Y`


---
## Decision stage: meta-control as expected value of control (EVC)


All variants implement a **meta-control policy**: on each trial, the agent can invest extra control (**proactive intention**, **reactive recall**, and sometimes **inhibition**) only if the **expected improvement in accuracy** is worth the **subjective cost**.

### Core principle (soft EVC choice rule)

For any control operation $u$,
$$
p(u \mid \text{available info})
\;=\;
\sigma\!\Big(
\underbrace{\mathbb{E}[\Delta \mathrm{acc}(u)\mid \text{info}] \cdot \alpha}_{\text{expected benefit (accuracy gain × value)}}
\;-\;
\underbrace{\mathrm{cost/bias}(u)}_{\text{effort, time pressure, thresholds}}
\Big),
$$
where $\sigma(\cdot)$ is a logistic mapping that turns **net benefit** into a probability (a noisy / soft-threshold decision rule).


### Why the benefit looks like $\mathbb{E}[\Delta \mathrm{acc}] \times \alpha$

Each control operation is justified by how much it improves accuracy **relative to habit**:
$$
\Delta \mathrm{acc}(u) \;=\; p_{\mathrm{correct},u} \;-\; p_{\mathrm{correct,habit}}.
$$

Because the agent does not know the exact upcoming trial type, it uses **only the information available at that stage**:

- **Cue stage (A vs B):** condition on the cue  
  $$
  \mathbb{E}[\Delta \mathrm{acc} \mid \mathrm{cue}]
  $$
- **Probe stage (X vs Y):** condition on the probe  
  $$
  \mathbb{E}[\Delta \mathrm{acc} \mid \mathrm{probe}]
  $$

The parameters $\alpha$ (e.g., $\alpha_{\mathrm{AX}}$ vs $\alpha$) scale accuracy gains into subjective “value” units: larger $\alpha$ means the agent values correctness more (or values specific high-stakes conditions more).



### Two benefit analyses: intention vs recall

They correspond to **two distinct control mechanisms**, available at **different times**, correcting **different failure modes**:

| Control operation | When chosen | What it fixes | Benefit term |
|---|---|---|---|
| Intention setting (proactive) | after cue | prevents habitual slips by biasing action in advance | $\mathbb{E}[\Delta \mathrm{acc}_{\mathrm{proactive}} \mid \mathrm{cue}]$ |
| Recall (reactive) | after probe | retrieves/uses contextual info to correct the response online | $\mathbb{E}[\Delta \mathrm{acc}_{\mathrm{recall}} \mid \mathrm{probe}]$ |

So the model evaluates both because they are different **cognitive actions** with different triggers, costs, and expected payoffs.


---
## Pipeline

#### 1) Intention setting (cue-based; proactive control)

$$
p(\text{intention}\mid \mathrm{cue})
=
\sigma\!\big(\mathrm{net\_benefit}_{\mathrm{intent}}(\mathrm{cue})\big)
$$

Implemented net benefits:

- cue $A$: `E_delta_acc_A*alphaAX + delta_t - gamma`  
- cue $B$: `E_delta_acc_B*alpha   + delta_t - gamma`

**Interpretation of extra terms (as used in code structure):**
- $\delta_t$ acts a stage-dependent bias 
    - interpreted as time/urgency, or a baseline preference for proactive preparation.
- $\gamma$ is a effort-aversion bias: larger $\gamma \Rightarrow$ less control engagement.


#### 2) Inhibition / control intensity selection (probe-based; only inhibition variants)

Choose intensity $c$ by maximizing **benefit − cost** over a grid of control intensities:

- benefit typically scales with $c$ (stronger inhibition/control reduces the impact of wrong habitual responding)
- cost is nonlinear:
$$
\mathrm{cost}(c) = \exp(\mathrm{control\_cost}\cdot |c|) - 1
$$

**So strong control is disproportionately expensive.**

This is meta-control over control strength, not just a binary “inhibit or not.”



#### 3) Recall decision (probe-based; reactive control)

$$
p(\text{recall}\mid \mathrm{probe})
=
\sigma\!\big(\mathrm{net\_benefit}_{\mathrm{recall}}(\mathrm{probe})\big)
$$

Implemented net benefits:

- probe $X$: `E_delta_acc_recall_X*alphaAX - delta_t - gamma`  
- probe $Y$: `E_delta_acc_recall_Y*alpha   - delta_t - gamma`

**Why the sign flips on $\delta_t$ here :**  
$\delta_t$ is used as a **relative bias** that pushes the policy toward one control mode versus the other (proactive intention vs reactive recall), encoding a tradeoff like “prepare early” vs “correct late.”

#### 4) Accuracy mixture

After choosing these control operations probabilistically, **each variant defines a different mixture** over branches (proactive vs recall vs inhibition vs habit) to compute `predicted_accs(1,s)`.

---

## Variable glossary by model (Lieder & Iwama, 2021)

**Model columns**:  
- **mDMC (Fig. 2)** = two independent meta-control decisions (set intention; recall rules)  
- **Extended + inhibition (Fig. 6a)** = adds probe-triggered inhibition control signal (can inhibit/boost intention)  
- **Exclusivity w/o inhibition (Fig. 6b)** = recall decision only if no intention was set  
- **Exclusivity + inhibition (Fig. 6c)** = can inhibit intention; if inhibited then habit governs response  

| Symbol / variable | Meaning (plain language) | Stage / conditioning | mDMC (Fig.2) | Extended + inhib (Fig.6a) | Exclusivity no inhib (Fig.6b) | Exclusivity + inhib (Fig.6c) | Notes / common MATLAB counterpart |
|---|---|---:|:---:|:---:|:---:|:---:|---|
| $cue$ | cue identity (A vs B) | cue stage | ✅ | ✅ | ✅ | ✅ | `cue` / `cues{s}` |
| $probe$ | probe identity (X vs Y; sometimes a NoGo probe in extensions) | probe stage | ✅ | ✅ | ✅ | ✅ | `probe` / `probes{s}` |
| $load$ | contextual / working-memory load (e.g., how many letters can serve as A-cue) | affects control success | ✅ | ✅ | ✅ | ✅ | `load` |
| $\theta$ | parameter vector (core parameters) | global | ✅ | ✅ | ✅ | ✅ | model params struct / vector |
| $I\in\{0,1\}$ | whether an intention was set at cue stage | cue→probe link | ✅ | ✅ | ✅ | ✅ | often maps to `p_intention` or latent `I` |
| $C\in\{0,1\}$ | whether the **choice** (intended response) is correct | outcome before motor slip | ✅ | ✅ | ✅ | ✅ | sometimes implicit; paper uses $C$ |
| $R\in\{0,1\}$ | whether the **button press** is correct after motor slip | final observed response | ✅ | ✅ | ✅ | ✅ | final predicted accuracy typically corresponds to $P(R{=}1)$ |
| $p_{\text{slip}}$ | probability of motor slip that flips correct/incorrect button press | response noise | ✅ | ✅ | ✅ | ✅ | `p_slip` |
| $u_+$ | subjective utility of correctly detecting an AX trial | incentive: accuracy (AX) | ✅ | ✅ | ✅ | ✅ | often corresponds to condition-specific reward weight (your code: `alphaAX`) |
| $u_-$ | subjective utility of correctly reporting “not AX” | incentive: accuracy (non-AX) | ✅ | ✅ | ✅ | ✅ | often corresponds to reward weight for non-AX (your code: `alpha`) |
| $u_{\Delta t}$ | subjective utility of responding quickly | incentive: speed | ✅ | ✅ | ✅ | ✅ | code uses `delta_t` as a bias-like term; paper uses $u_{\Delta t}$ in benefit computations |
| $\lambda$ | intensity of deleterious effect of load on controlled processing | control success penalty | ✅ | ✅ | ✅ | ✅ | `lambda` |
| $\gamma$ | cost of setting/maintaining intentions (and used as recall cost term in the paper’s recall rule) | effort/threshold cost | ✅ | ✅ | ✅ | ✅ | `gamma` (and/or embedded as constant subtraction in net-benefit) |
| $P(I{=}1\mid cue,\theta,load)$ | probability to set intention given cue | cue meta-control | ✅ | ✅ | ✅ | ✅ | code: `p_intention = sigmoid(net_benefit_intent)` |
| $c_{\text{recall}}\in\{0,1\}$ | whether the model recalls cue+rules at probe stage (“reactive control”) | probe meta-control | ✅ | ✅ | ⚠️ (only if $I{=}0$) | ⚠️ (only if $I{=}0$) | code: `p_recall = sigmoid(net_benefit_recall)` |
| $P(c_{\text{recall}}{=}1\mid probe,u_+,u_-,\gamma,u_{\Delta t},load)$ | probability of recalling rules given probe | probe meta-control | ✅ | ✅ | ⚠️ | ⚠️ | in code this corresponds to probe-based recall decision |
| $CR$ | event that recall+rule application succeeds (reactive correctness) | accuracy component | ✅ | ✅ | ✅ (when recall occurs) | ✅ (when recall occurs) |  code groups this into `p_correct_recall` |
| $CM$ | accuracy under automaticity / probability matching (habit-like responding) | accuracy component | ✅ | ✅ | ✅ | ✅ | code: `p_correct_habit` |
| $CI$ | event that proactively formed intention is correct | accuracy component | ✅ (used when $I{=}1$) | ✅ | ✅ | ✅ | code: `p_correct_proactive` |
| $c$ | probe-triggered control signal intensity that boosts ($c>0$) or inhibits ($c<0$) intention | inhibition meta-control | ❌ | ✅ | ❌ | ✅ |  code: `control_intensities` grid; `optimal_c` |
| $c_0$ | default baseline weight of intention before modulation (e.g., a fixed baseline in the paper) | inhibition baseline | ❌ | ✅ | ❌ | ✅ |  code often uses `random_inhibition` / baseline mixture weights |
| $\delta$ | control-cost parameter controlling how fast inhibition cost grows with $c$ | inhibition cost | ❌ | ✅ | ❌ | ✅ |  code: `control_cost` corresponds to paper’s $\delta$ |
| $cost(c,\delta)$ | cost of applying control signal $c$ | inhibition cost function | ❌ | ✅ | ❌ | ✅ | $cost(c,\delta)=\exp(\delta |c|)-1$ |
| $c^\star$ | optimal control signal chosen by maximizing expected benefit minus cost | inhibition policy | ❌ | ✅ | ❌ | ✅ | code: `optimal_c = argmax(...)` |
| $Inhibit\in\{0,1\}$ | whether intention is inhibited at probe stage | inhibition gate | ❌ | ✅ | ❌ | ✅ | code: `p_inhibition` |
| $P(Inhibit\mid probe,load,\theta)$ | probability to inhibit (depends on $c^\star$, $c_0$, and load) | inhibition gate | ❌ | ✅ | ❌ | ✅ | in the paper: often shaped like $(1-(c_0+c))(1-\lambda\cdot load)$; code uses a closely related form |


## Accuracy-mixture by model

Across this model family, the *final predicted accuracy* for each trial type $s$ is a **mixture over control branches** (proactive intention, reactive recall, inhibition, habit). 

In the MATLAB scripts, branch-specific “accuracies” are computed as vectors like:
- `p_correct_proactive(s)`  (≈ $P(R{=}1 \mid \text{intention enacted})$)
- `p_correct_recall(s)`     (≈ $P(R{=}1 \mid \text{recall \& apply rules})$)
- `p_correct_habit(s)`      (≈ $P(R{=}1 \mid \text{habit / probability matching})$)

`predicted_accs(1,s)` is the **weighted sum of those branch accuracies**, where weights are the meta-control probabilities (`p_intention`, `p_recall`, `p_inhibition`) *and their complements*.

- MATLAB `p_correct_*` terms already include the motor-slip mixture via `p_slip`, so `predicted_accs` is effectively $P_{\text{model}}(R{=}1)$ directly
    - indicated by the repeated pattern `...*(1-p_slip) + p_slip*...` inside the definitions of `p_correct_habit`, `p_correct_proactive`, `p_correct_recall`.
- In contrast, the paper writes $P(C{=}1)$ first, then converts to $P(R{=}1)$).  

| MATLAB script | Paper model / figure match | Branch mixture (conceptual) for $P(R{=}1\mid s)$ | MATLAB `predicted_accs` line (exact, with line #) |
|---|---|---|---|
| `metaControlModelRecallOverride.m` | **mDMC / Fig. 2** (recall can override even if intention exists) | 4-branch mixture: <br>1) $I{=}1,\;Recall{=}1 \Rightarrow$ recall <br>2) $I{=}1,\;Recall{=}0 \Rightarrow$ proactive <br>3) $I{=}0,\;Recall{=}1 \Rightarrow$ recall <br>4) $I{=}0,\;Recall{=}0 \Rightarrow$ habit | **L114–117**: <br>`predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...`<br>`    p_intention*(1-p_recall)*p_correct_proactive(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |
| `metaControlModelRecallOverrideNoGo.m` | same architecture, with NoGo trial types in $s$ | same 4-branch mixture | **L115–118**: <br>`predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...`<br>`    p_intention*(1-p_recall)*p_correct_proactive(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |
| `metaControlModelInhibitionRecall.m` | **Extended + inhibition / Fig. 6a** (inhibit intention, then optionally recall) | 5-branch mixture: <br>1) $I{=}1,\;Inhib{=}0 \Rightarrow$ proactive <br>2) $I{=}1,\;Inhib{=}1,\;Recall{=}1 \Rightarrow$ recall <br>3) $I{=}1,\;Inhib{=}1,\;Recall{=}0 \Rightarrow$ habit <br>4) $I{=}0,\;Recall{=}1 \Rightarrow$ recall <br>5) $I{=}0,\;Recall{=}0 \Rightarrow$ habit | **L124–128**: <br>`predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...`<br>`    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...`<br>`    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |
| `metaControlModelInhibitionRecallNoGo.m` | same as above, with NoGo trial types in $s$ | same 5-branch mixture | **L138–142**: <br>`predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...`<br>`    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...`<br>`    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |
| `metaControlModelNoInhibitionNoGo.m` | **Exclusivity w/o inhibition / Fig. 6b** (recall only when no intention) | 3-branch mixture: <br>1) $I{=}1 \Rightarrow$ proactive <br>2) $I{=}0,\;Recall{=}1 \Rightarrow$ recall <br>3) $I{=}0,\;Recall{=}0 \Rightarrow$ habit | **L113–115**: <br>`predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |
| `metaControlModelNoInhibition.m` | intended to match Fig. 6b, but omits the final “habit when $I{=}0, Recall{=}0$” term | (expected) same 3-branch mixture as above | **L106–108**: <br>`predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...` |
| `metaControlModelInhibitionHabitualNoGo.m` | **Exclusivity + inhibition / Fig. 6c** (if intention inhibited → habit, no recall) | 4-branch mixture: <br>1) $I{=}1,\;Inhib{=}0 \Rightarrow$ proactive <br>2) $I{=}1,\;Inhib{=}1 \Rightarrow$ habit <br>3) $I{=}0,\;Recall{=}1 \Rightarrow$ recall <br>4) $I{=}0,\;Recall{=}0 \Rightarrow$ habit | **L131–134**: <br>`predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...`<br>`    p_intention*p_inhibition*p_correct_habit(s) + ...`<br>`    (1-p_intention)*p_recall*p_correct_recall(s) + ...`<br>`    (1-p_intention)*(1-p_recall)*p_correct_habit(s);` |

- $p_{\text{intention}}$ / `p_intention`: probability of setting an intention from the cue-stage meta-control policy.
- $p_{\text{recall}}$ / `p_recall`: probability of engaging recall (reactive control) at probe stage (conditioning on probe).
- $p_{\text{inhibition}}$ / `p_inhibition`: probability of inhibiting a currently active intention at probe stage (only relevant when $I{=}1$).
- `p_correct_proactive(s)`: branch accuracy if intention is enacted (includes $p_{\text{slip}}$).
- `p_correct_recall(s)`: branch accuracy if recall/rule use happens (includes $p_{\text{slip}}$).
- `p_correct_habit(s)`: branch accuracy under habit / probability matching (includes $p_{\text{slip}}$).
- $load, \lambda$: load and load-penalty terms that reduce controlled accuracies (paper sometimes writes this as a subtractive term; the MATLAB scripts bake it into `p_correct_*`).

---
## 1) NoInhibition (Go-only)

**MATLAB call:** `metaControlModel(alpha, alphaAX, gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
metaControlModel(alpha, alphaAX, gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

% alpha = reward 
% gamma = cost of setting intention and recalling
% delta = control cost
% delta_t = reward for fast responses
% lambda = cognitive load interference

cues={'A','A','B','B'};
probes={'X','Y','X','Y'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);

p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pBX), 0, 1-pBX/(pAX+pBX),0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0]; 
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)]; 
    
else
    with_A=[1,1,0,0];
    with_B=[0,0,1,1];
    with_X=[1,0,1,0];
    with_Y=[0,1,0,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y;    
    
end

p_correct_proactive = [pAX/(pAX+pAY) * (1 - lambda*(load-1)),...
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)) ... 
    ]*(1-p_random_error)...
    +p_random_error*[1-(pAX/(pAX+pAY))* (1 - lambda*(load-1)),... 
    pAX/(pAX+pAY)* (1 - lambda*(load-1)),... 
    0* (1 - lambda*(load-1)), ... 
    0* (1 - lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);


for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
     elseif strcmp(cue,'B') 
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
  
    % Step 2: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
    
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = 0; % optimal_cs(1,s) = optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    
    % Predicted accuracies
    predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...

       
end

end
```


---
## NoInhibitionNoGo (Go+NoGo)

**MATLAB call:** `metaControlModelNoGo(alpha,alphaAX,gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY, AN, BN


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Extends the trial space with **NoGo probe `N`** (AN/BN), and adjusts proactive correctness terms accordingly.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
    metaControlModelNoGo(alpha,alphaAX,gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

cues={'A','A','B','B','A','B'};
probes={'X','Y','X','Y','N','N'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);
pAN = p_cue_probe(5);
pBN = p_cue_probe(6);

% All sensitive to the probe
p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1, 1, 1]*(1-p_random_error) ...
    + p_random_error*[1-pAX/(pAX+pBX),0,1-pBX/(pAX+pBX),0,0,0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0]; 
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)]; 
    
 
else
    with_A=[1,1,0,0,1,0];
    with_B=[0,0,1,1,0,1];
    with_X=[1,0,1,0,0,0];
    with_Y=[0,1,0,1,0,0];
    with_N=[0,0,0,0,1,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    p_N=dot(p_cue_probe,with_N);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y; 
    p_given_N=p_cue_probe.*with_N/p_N;
  
end

p_correct_proactive =  [pAX/(pAX+pAY) * (1 - lambda*(load-1)),... %AX
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),...%AY
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BX
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ...%BY
    pAN/(pAX+pAY+pAN) * (1 - lambda*(load-1)), ... %AN
    pBN/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BN
    ]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-pAN/(pAX+pAY+pAN) * (1 + lambda*(load-1)),...
    1-pBN/(pBX+pBY+pBN) * (1 + lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);
E_delta_acc_N = dot(delta_acc,p_given_N);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);
E_delta_acc_recall_N = dot(delta_acc_recall,p_given_N);

for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma; 
    elseif strcmp(cue,'B')
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
        
    % Step 2: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
     elseif strcmp(probe,'N')
        net_benefit_recall = E_delta_acc_recall_N*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
   
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = 0; %optimal_cs(1,s) = optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    
    % Predicted accuracies
    predicted_accs(1,s) = p_intention*p_correct_proactive(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);
    
   
end

end
```


---
## InhibitionRecall (Go-only)

**MATLAB call:** `metaControlModelInhibitionRecall(alpha, alphaAX, gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Uses `p_inhibition` as an explicit gate between proactive/intention execution and alternative routes.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...
    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 2: probe-based inhibition/control intensity choice
  control_intensities = linspace(-random_inhibition, 1-random_inhibition, 101)
  benefit_c(c) = (delta_t + E_delta_acc_probe * reward_weight_probe) * c
  cost(c) = exp(control_cost * abs(c)) - 1
  optimal_c = argmax_c (benefit_c(c) - cost(c))
  p_inhibition = (1 - (random_inhibition + optimal_c)) * (1 - lambda*(load-1))

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
metaControlModelInhibitionRecall(alpha, alphaAX, gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

% alpha = reward 
% gamma = cost of setting intention and recalling
% delta = control cost
% delta_t = reward for fast responses
% lambda = cognitive load interference

cues={'A','A','B','B'};
probes={'X','Y','X','Y'};

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);

p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1]*(1-p_slip)...
    +p_slip*[1-pAX/(pAX+pBX), 0, 1-pBX/(pAX+pBX),0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0]; 
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)]; 
    
else
    with_A=[1,1,0,0];
    with_B=[0,0,1,1];
    with_X=[1,0,1,0];
    with_Y=[0,1,0,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y;    
    
end

p_correct_proactive = [pAX/(pAX+pAY) * (1 - lambda*(load-1)),...
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)) ... 
    ]*(1-p_slip)...
    +p_slip*[1-(pAX/(pAX+pAY))* (1 - lambda*(load-1)),... 
    pAX/(pAX+pAY)* (1 - lambda*(load-1)),... 
    0* (1 - lambda*(load-1)), ... 
    0* (1 - lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_slip)...
+ p_slip*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);


for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
       net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
     elseif strcmp(cue,'B') 
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
    %Step 2: boosting or inhibiting the intention
    control_intensities=linspace(-random_inhibition,1-random_inhibition,101);
    if strcmp(probe,'X')
        benefit_c = @(c) (delta_t+E_delta_acc_X*alphaAX)*c;
     elseif strcmp(probe,'Y')
        benefit_c = @(c) (delta_t+E_delta_acc_Y*alpha)*c;
    end
    cost = @(c) exp(control_cost*abs(c)) - 1;
    
    pos=argmax( benefit_c(control_intensities) -cost(control_intensities) );
    optimal_c=control_intensities(pos);
    
    p_inhibition = (1 - (random_inhibition + optimal_c)) * (1 - lambda*(load-1));
    
    % Step 3: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
    
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    p_inhibitions(1,s) = p_inhibition;
    
    % Predicted accuracies
        %1. p_intention*(1-p_inhibition)*p_cue_probe(s)
        %2. p_intention*p_inhibition*p_recall*p_correct_recall
        %3. p_intention*p_inhibition*(1-p_recall)*p_habit(s)
        %4. (1-p_intention)*p_recall*p_correct_recall
        %5. (1-p_intention)*(1-p_recall)*p_habit(s)
        
    predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...
    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);

       
end

end
```


---
## InhibitionRecallNoGo (Go+NoGo)

**MATLAB call:** `metaControlModelNoGo(alpha,alphaAX,gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs,p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY, AN, BN


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Uses `p_inhibition` as an explicit gate between proactive/intention execution and alternative routes.
- Extends the trial space with **NoGo probe `N`** (AN/BN), and adjusts proactive correctness terms accordingly.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...
    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 2: probe-based inhibition/control intensity choice
  control_intensities = linspace(-random_inhibition, 1-random_inhibition, 101)
  benefit_c(c) = (delta_t + E_delta_acc_probe * reward_weight_probe) * c
  cost(c) = exp(control_cost * abs(c)) - 1
  optimal_c = argmax_c (benefit_c(c) - cost(c))
  p_inhibition = (1 - (random_inhibition + optimal_c)) * (1 - lambda*(load-1))

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs,p_intentions, optimal_cs, p_recalls] = ...
    metaControlModelNoGo(alpha,alphaAX,gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

% alpha = reward 
% gamma = cost of setting intention and recalling
% delta = control cost
% delta_t = reward for fast responses
% lambda = cognitive load interference

cues={'A','A','B','B','A','B'};
probes={'X','Y','X','Y','N','N'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);
pAN = p_cue_probe(5);
pBN = p_cue_probe(6);
ptotal = pAX + pAY + pBX + pBY + pAN + pBN;

% All sensitive to the probe
p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1, 1, 1]*(1-p_random_error) ...
    + p_random_error*[1-pAX/(pAX+pBX),0,1-pBX/(pAX+pBX),0,0,0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0];
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)];    
    
else
    with_A=[1,1,0,0,1,0];
    with_B=[0,0,1,1,0,1];
    with_X=[1,0,1,0,0,0];
    with_Y=[0,1,0,1,0,0];
    with_N=[0,0,0,0,1,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    p_N=dot(p_cue_probe,with_N);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y; 
    p_given_N=p_cue_probe.*with_N/p_N;
    
end


p_correct_proactive =  [pAX/(pAX+pAY) * (1 - lambda*(load-1)),... % AX
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)), ...% AY
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... % BX
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ...% BY
    pAN/(pAX+pAY+pAN) * (1 - lambda*(load-1)), ... % AN
    pBN/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... % BN
    ]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-pAN/(pAX+pAY+pAN) * (1 + lambda*(load-1)),...
    1-pBN/(pBX+pBY+pBN) * (1 + lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);
E_delta_acc_N = dot(delta_acc,p_given_N);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);
E_delta_acc_recall_N = dot(delta_acc_recall,p_given_N);

for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
    elseif strcmp(cue,'B')
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
    % Step 2: boosting or inhibiting the intention
    control_intensities=linspace(-random_inhibition,1-random_inhibition,101);
    if strcmp(probe,'X')
        benefit = @(c) (delta_t+E_delta_acc_X*alphaAX)*c; 
     elseif strcmp(probe,'Y')
        benefit = @(c) (delta_t+E_delta_acc_Y*alpha)*c; 
     elseif strcmp(probe,'N')
        benefit = @(c) (0.5-c)*alpha;
    end
    cost = @(c) exp(control_cost*abs(c)) - 1;
    
    pos=argmax( benefit(control_intensities) -cost(control_intensities) );
    optimal_c=control_intensities(pos);
    
    p_inhibition = (1 - (random_inhibition + optimal_c)) * (1-lambda*load);
    
    % Step 3: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
     elseif strcmp(probe,'N')
        net_benefit_recall = E_delta_acc_recall_N*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
   
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    p_inhibitions(1,s) = p_inhibition;
    

    % Predicted accuracies
    predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
    p_intention*p_inhibition*p_recall*p_correct_recall(s) + ...
    p_intention*p_inhibition*(1-p_recall)*p_correct_habit(s) + ...
    (1-p_intention)*p_recall*p_correct_recall(s) + ...
    (1-p_intention)*(1-p_recall)*p_correct_habit(s);

   
end

end
```


---
## InhibitionHabitualNoGo (Go+NoGo)

**MATLAB call:** `metaControlModelNoGo(alpha,alphaAX,gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY, AN, BN


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Uses `p_inhibition` as an explicit gate between proactive/intention execution and alternative routes.
- Extends the trial space with **NoGo probe `N`** (AN/BN), and adjusts proactive correctness terms accordingly.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
        p_intention*p_inhibition*p_correct_habit(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 2: probe-based inhibition/control intensity choice
  control_intensities = linspace(-random_inhibition, 1-random_inhibition, 101)
  benefit_c(c) = (delta_t + E_delta_acc_probe * reward_weight_probe) * c
  cost(c) = exp(control_cost * abs(c)) - 1
  optimal_c = argmax_c (benefit_c(c) - cost(c))
  p_inhibition = (1 - (random_inhibition + optimal_c)) * (1 - lambda*(load-1))

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
    metaControlModelNoGo(alpha,alphaAX,gamma,load,control_cost,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

cues={'A','A','B','B','A','B'};
probes={'X','Y','X','Y','N','N'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);
pAN = p_cue_probe(5);
pBN = p_cue_probe(6);
ptotal = pAX + pAY + pBX + pBY + pAN + pBN;

% All sensitive to the probe
p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1, 1, 1]*(1-p_random_error) ...
    + p_random_error*[1-pAX/(pAX+pBX),0,1-pBX/(pAX+pBX),0,0,0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0]; 
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)]; 

else
    with_A=[1,1,0,0,1,0];
    with_B=[0,0,1,1,0,1];
    with_X=[1,0,1,0,0,0];
    with_Y=[0,1,0,1,0,0];
    with_N=[0,0,0,0,1,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    p_N=dot(p_cue_probe,with_N);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y; 
    p_given_N=p_cue_probe.*with_N/p_N;
    
end


p_correct_proactive =  [pAX/(pAX+pAY) * (1 - lambda*(load-1)),... %AX
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),...%AY
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BX
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ...%BY
    pAN/(pAX+pAY+pAN) * (1 - lambda*(load-1)), ... %AN
    pBN/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BN
    ]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-pAN/(pAX+pAY+pAN) * (1 + lambda*(load-1)),...
    1-pBN/(pBX+pBY+pBN) * (1 + lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);
E_delta_acc_N = dot(delta_acc,p_given_N);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);
E_delta_acc_recall_N = dot(delta_acc_recall,p_given_N);

for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
    elseif strcmp(cue,'B')
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
    % Step 2: boosting or inhibiting the intention
    control_intensities=linspace(-random_inhibition,1-random_inhibition,101);
    if strcmp(probe,'X')
        benefit = @(c) (delta_t+E_delta_acc_X*alphaAX)*c; 
     elseif strcmp(probe,'Y')
        benefit = @(c) (delta_t+E_delta_acc_Y*alpha)*c;
     elseif strcmp(probe,'N')
        benefit = @(c) (0.5-c)*alpha;
    end
    cost = @(c) exp(control_cost*abs(c)) - 1;
    
    pos=argmax( benefit(control_intensities) -cost(control_intensities) );
    optimal_c=control_intensities(pos);
    
    p_inhibition = (1 - (random_inhibition + optimal_c)) * (1-lambda*load);
    
    % Step 3: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
     elseif strcmp(probe,'N')
        net_benefit_recall = E_delta_acc_recall_N*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
   
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    p_inhibitions(1,s) = p_inhibition;
    
    % Predicted accuracies
    predicted_accs(1,s) = p_intention*(1-p_inhibition)*p_correct_proactive(s) + ...
        p_intention*p_inhibition*p_correct_habit(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);

   
end

end
```


---
## RecallOverride (Go-only)

**MATLAB call:** `metaControlModelRecallOverride(alpha, alphaAX, gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Uses `p_inhibition` as an explicit gate between proactive/intention execution and alternative routes.
- Uses a **recall override mixture**: when recall occurs, it can supersede proactive/intention outcomes (see mixture below).

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...
        p_intention*(1-p_recall)*p_correct_proactive(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
metaControlModelRecallOverride(alpha, alphaAX, gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

% alpha = reward 
% gamma = cost of setting intention and recalling
% delta = control cost
% delta_t = reward for fast responses
% lambda = cognitive load interference

cues={'A','A','B','B'};
probes={'X','Y','X','Y'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);

p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pBX), 0, 1-pBX/(pAX+pBX),0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0]; 
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)]; 
    
    
else
    with_A=[1,1,0,0];
    with_B=[0,0,1,1];
    with_X=[1,0,1,0];
    with_Y=[0,1,0,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y;    
    
    
end

p_correct_proactive = [pAX/(pAX+pAY) * (1 - lambda*(load-1)),...
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)),... 
    1 * (1 - lambda*(load-1)) ... 
    ]*(1-p_random_error)...
    +p_random_error*[1-(pAX/(pAX+pAY))* (1 - lambda*(load-1)),...
    pAX/(pAX+pAY)* (1 - lambda*(load-1)),...
    0* (1 - lambda*(load-1)), ... 
    0* (1 - lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);


for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
     elseif strcmp(cue,'B') 
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
    
    % Step 2: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
    
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = 0; %optimal_c;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;
    
    % Predicted accuracies
        %1. p_intention*(1-p_inhibition)*p_cue_probe(s)
        %2. p_intention*p_inhibition*p_recall*p_correct_recall
        %3. p_intention*p_inhibition*(1-p_recall)*p_habit(s)
        %4. (1-p_intention)*p_recall*p_correct_recall
        %5. (1-p_intention)*(1-p_recall)*p_habit(s)
        
    predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...
        p_intention*(1-p_recall)*p_correct_proactive(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);
 
   
       
end

end
```


---
## RecallOverrideNoGo (Go+NoGo)

**MATLAB call:** `metaControlModelRecallOverrideNoGo(alpha,alphaAX,gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)`

**Outputs:** `predicted_accs, p_intentions, optimal_cs, p_recalls`

**Trial types:** AX, AY, BX, BY, AN, BN


### Implementation deltas
- Implements a **control intensity search** over `control_intensities` to pick `optimal_c` (via `argmax`), then derives `p_inhibition`.
- Uses a **recall override mixture**: when recall occurs, it can supersede proactive/intention outcomes (see mixture below).
- Extends the trial space with **NoGo probe `N`** (AN/BN), and adjusts proactive correctness terms accordingly.

### Predicted accuracy mixture (as implemented)
```matlab
predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...
        p_intention*(1-p_recall)*p_correct_proactive(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);
```


**Pseudocode mirroring the implementation**

```text
for each trial type s in {AX, AY, BX, BY[, AN, BN]}:
  cue = cues[s]; probe = probes[s]

  # Shared: compute p_correct_habit(s), p_correct_proactive(s), p_correct_recall(s)
  # Shared: compute expected benefits E_delta_acc_{cue/probe}, E_delta_acc_recall_{probe}

  # Step 1: intention setting (cue-based)
  net_benefit = (E_delta_acc_cue * reward_weight) + delta_t - gamma
  p_intention = sigmoid(net_benefit)

  # Step 3: recall decision (probe-based)
  net_benefit_recall = (E_delta_acc_recall_probe * reward_weight_probe) - delta_t - gamma
  p_recall = sigmoid(net_benefit_recall)

  # Step 4: mix branches to compute predicted_accs(1,s)
  predicted_accs(1,s) = mixture_over_branches(p_intention, p_inhibition, p_recall,
                                             p_correct_proactive(s), p_correct_recall(s), p_correct_habit(s))
end
```

### Original MATLAB source (verbatim)
```matlab
function [predicted_accs, p_intentions, optimal_cs, p_recalls] = ...
    metaControlModelRecallOverrideNoGo(alpha,alphaAX,gamma,load,delta_t,lambda,m,random_inhibition,p_cue_probe,p_slip)

cues={'A','A','B','B','A','B'};
probes={'X','Y','X','Y','N','N'};

p_random_error=p_slip;

pAX = p_cue_probe(1);
pAY = p_cue_probe(2);
pBX = p_cue_probe(3);
pBY = p_cue_probe(4);
pAN = p_cue_probe(5);
pBN = p_cue_probe(6);

% All sensitive to the probe
p_correct_habit=[pAX/(pAX+pBX), 1, pBX/(pAX+pBX), 1, 1, 1]*(1-p_random_error) ...
    + p_random_error*[1-pAX/(pAX+pBX),0,1-pBX/(pAX+pBX),0,0,0]; 

if not(exist('p_cue_probe','var'))
    p_given_A=[pAX/(pAX+pAY), pAY/(pAX+pAY), 0, 0];
    p_given_B=[0, 0, pBX/(pBX+pBY), pBX/(pBX+pBY)];
    p_given_X=[pAX/(pAX+pBX), 0, pBX/(pAX+pBX), 0];
    p_given_Y=[0, pAY/(pAY+pBY), 0, pBY/(pAY+pBY)];  

    
else
    with_A=[1,1,0,0,1,0];
    with_B=[0,0,1,1,0,1];
    with_X=[1,0,1,0,0,0];
    with_Y=[0,1,0,1,0,0];
    with_N=[0,0,0,0,1,1];
    
    p_A=dot(p_cue_probe,with_A);
    p_B=dot(p_cue_probe,with_B);
    p_X=dot(p_cue_probe,with_X);
    p_Y=dot(p_cue_probe,with_Y);
    p_N=dot(p_cue_probe,with_N);
    
    p_given_A=p_cue_probe.*with_A/p_A;
    p_given_B=p_cue_probe.*with_B/p_B;
    p_given_X=p_cue_probe.*with_X/p_X;
    p_given_Y=p_cue_probe.*with_Y/p_Y; 
    p_given_N=p_cue_probe.*with_N/p_N;
    
end


p_correct_proactive =  [pAX/(pAX+pAY) * (1 - lambda*(load-1)),... %AX
    (1-pAX/(pAX+pAY)) * (1 - lambda*(load-1)),...%AY
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BX
    (pBX+pBY)/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ...%BY
    pAN/(pAX+pAY+pAN) * (1 - lambda*(load-1)), ... %AN
    pBN/(pBX+pBY+pBN) * (1 - lambda*(load-1)), ... %BN
    ]*(1-p_random_error)...
    +p_random_error*[1-pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    pAX/(pAX+pAY) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-(pBX+pBY)/(pBX+pBY+pBN) * (1 + lambda*(load-1)),...
    1-pAN/(pAX+pAY+pAN) * (1 + lambda*(load-1)),...
    1-pBN/(pBX+pBY+pBN) * (1 + lambda*(load-1))];

% Benefit of setting and intention
delta_acc = p_correct_proactive-p_correct_habit;

% p(correct response | recall) 
p_correct_recall = [m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1)), m * (1 - lambda*(load-1))]*(1-p_random_error)...
+ p_random_error*[1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1))), 1-(m * (1 - lambda*(load-1)))];

% Benefit of recalling
delta_acc_recall = p_correct_recall-p_correct_habit;

% Net benefit
E_delta_acc_A = dot(delta_acc,p_given_A);
E_delta_acc_B = dot(delta_acc,p_given_B);
E_delta_acc_X = dot(delta_acc,p_given_X);
E_delta_acc_Y = dot(delta_acc,p_given_Y);
E_delta_acc_N = dot(delta_acc,p_given_N);

E_delta_acc_recall_A = dot(delta_acc_recall,p_given_A);
E_delta_acc_recall_B = dot(delta_acc_recall,p_given_B);
E_delta_acc_recall_X = dot(delta_acc_recall,p_given_X);
E_delta_acc_recall_Y = dot(delta_acc_recall,p_given_Y);
E_delta_acc_recall_N = dot(delta_acc_recall,p_given_N);

for s=1:numel(cues)
    
    cue = cues{s}; probe=probes{s};
    
    %Step 1: intention setting
    if strcmp(cue,'A')
        net_benefit = E_delta_acc_A*alphaAX+delta_t-gamma;
    elseif strcmp(cue,'B')
        net_benefit = E_delta_acc_B*alpha+delta_t-gamma;
    end
    p_intention = sigmoid(net_benefit);
    
    
    % Step 2: Decide to recall or not 
    if strcmp(probe,'X')
        net_benefit_recall = E_delta_acc_recall_X*alphaAX-delta_t-gamma;
     elseif strcmp(probe,'Y')
        net_benefit_recall = E_delta_acc_recall_Y*alpha-delta_t-gamma;
     elseif strcmp(probe,'N')
        net_benefit_recall = E_delta_acc_recall_N*alpha-delta_t-gamma;
    end
    p_recall = sigmoid(net_benefit_recall);
   
    % Cost-Benefit analysis parameters for each trial type
    optimal_cs(1,s) = 0;
    p_intentions(1,s) = p_intention;
    p_recalls(1,s) = p_recall;

    % Predicted accuracies
    predicted_accs(1,s) = p_intention*p_recall*p_correct_recall(s) + ...
        p_intention*(1-p_recall)*p_correct_proactive(s) + ...
        (1-p_intention)*p_recall*p_correct_recall(s) + ...
        (1-p_intention)*(1-p_recall)*p_correct_habit(s);

   
end

end
```


---
## Cross-model differences that matter for interpretation

### Functional role of “inhibition”
- **NoInhibition / RecallOverride**: no explicit `p_inhibition`. The policy is essentially cue-based intention plus a probe-based recall option (with different mixtures across these two).
- **InhibitionRecall**: introduces an explicit probe-based inhibition gate (parameterized by `control_cost`, `random_inhibition`, and `optimal_c`), and uses recall as a *fallback* when inhibited.
- **InhibitionHabitualNoGo**: also uses `p_inhibition`, but uses it to route inhibited trials back toward the habitual policy (not recall) in the implemented mixture.

### `Control_cost` in inhibition models
In the inhibition models, `control_cost` shapes how steeply the cost of `|c|` grows:
- larger `control_cost` → strong penalty for extreme control intensities → smaller magnitude `optimal_c` (closer to 0).
- the code uses an exponential cost: `exp(control_cost*abs(c)) - 1`.

### NoGo extension changes proactive “correctness”
The NoGo variants redefine the proactive correctness terms for `BX/BY/BN` (and for `AN`) to incorporate `pBN` / `pAN` normalizations. 
- This means proactive control is no longer “always correct on BY” in the same way as Go-only variants;
- Instead, it depends on the **base-rates** involving the NoGo probe.

### ?
`metaControlModelNoInhibition.m` ends while building `predicted_accs(1,s)` (it has a dangling `+ ...` continuation). If you intend to run these models, that file needs to be completed (likely by adding habitual fallback terms analogous to other variants).
